# Week 4 Tutorial: Cleaning, Feature Engineering, and Simple Linear Regression

In [53]:
import pandas as pd
import matplotlib.pyplot as plt

### The Dataset

In [83]:
mock_df = pd.read_csv('mock_ord_departures.csv')
print(mock_df.shape)
mock_df.head()

(240, 8)


,observation_time,scheduled_departures,wind_speed,visibility,temperature,precipitation,weather_condition,avg_departure_delay_min
0,2024-01-09 06:00,37,19 mph,4.1 mi,26°F,0.03 in,Snow,47.9
1,2024-01-09 09:00,57,8 mph,10.0 mi,46°F,Trace,Clear,23.8
2,2024-01-09 12:00,41,20 mph,7.9 mi,34°F,0.00 in,Cloudy,27.4
3,2024-01-09 15:00,50,13 mph,9.0 mi,32°F,0.00 in,Cloudy,22.0
4,2024-01-09 18:00,57,20 mph,9.4 mi,36°F,NaN,Cloudy,25.8


Let avg_departure_delay_min be the response variable and all other columns the explanatory variables. If we're trying to use these variables to predict avg_departure_delay_min, what are some issues you notice?

##### Answers

1. Dates are in a difficult format and likely not datetime objects.
2. Some values have units (strings).
3. Some values are NaN/need interpolation.
4. Some variables are clearly important but categorical.

### Let's fix it

In [84]:
mock_df.dtypes

observation_time            object
scheduled_departures         int64
wind_speed                  object
visibility                  object
temperature                 object
precipitation               object
weather_condition           object
avg_departure_delay_min    float64
dtype: object

In [85]:
# datetime issue
mock_df['observation_time'] = pd.to_datetime(mock_df['observation_time'])

In [88]:
# unit issue
mock_df['wind_speed'] = mock_df['wind_speed'].str.replace(' mph', '')
mock_df['visibility'] = mock_df['visibility'].str.replace(' mi', '')
mock_df['precipitation'] = mock_df['precipitation'].str.replace(' in','')
mock_df['temperature'] = mock_df['temperature'].str.replace('°F', '')
# how would we handle temperature?

In [89]:
mock_df.head()

,observation_time,scheduled_departures,wind_speed,visibility,temperature,precipitation,weather_condition,avg_departure_delay_min
0,2024-01-09 06:00:00,37,19,4.1,26,0.03,Snow,47.9
1,2024-01-09 09:00:00,57,8,10.0,46,Trace,Clear,23.8
2,2024-01-09 12:00:00,41,20,7.9,34,0.00,Cloudy,27.4
3,2024-01-09 15:00:00,50,13,9.0,32,0.00,Cloudy,22.0
4,2024-01-09 18:00:00,57,20,9.4,36,NaN,Cloudy,25.8


In [38]:
# categorical variables

print(mock_df['weather_condition'].unique())
mock_df['weather_condition'] = pd.get_dummies(mock_df['weather_condition'], drop_first=True)

# this doesn't really show up in our actual dataset, so we don't really have to worry about it

array(['Snow', 'Clear', 'Cloudy', 'Fog', 'snow', ' Clear ',
       'Thunderstorm', 'Rain', 'cloudy', ' Cloudy ', ' Fog ', 'clear'],
      dtype=object)

In [92]:
# Null values
mock_df['precipitation'].isnull().sum() / len(mock_df['precipitation'])
mock_df['precipitation'] = mock_df['precipitation'].dropna()

# let's drop

In [104]:
mock_df = mock_df.dropna(subset=[
    'wind_speed',
    'visibility',
    'precipitation',
    'avg_departure_delay_min'
])

In [105]:
mock_df['wind_speed'].isnull().sum()

np.int64(0)

In [109]:
mock_df['wind_speed'] = mock_df['wind_speed'].astype(int)
mock_df['temperature'] = mock_df['temperature'].astype(int)

### Feature Engineering

Could making our own variables help predict delays? Let's try to list some examples.

##### Answers

1. day_of_week
2. is_weekend
3. high_wind
4. low_visibility

### Let's engineer

In [111]:
# day_of_week and is_weekend
mock_df['day_of_week'] = mock_df['observation_time'].dt.day_name()
print(mock_df['day_of_week'].unique())
# we'd then convert to dummy variables

mock_df['is_weekend'] = mock_df['day_of_week'].isin(['Saturday', 'Sunday'])
print(mock_df['is_weekend'])

['Tuesday' 'Wednesday' 'Thursday' 'Friday' 'Saturday' 'Sunday' 'Monday']
0      False
1      False
2      False
3      False
5      False
       ...  
235    False
236    False
237    False
238    False
239    False
Name: is_weekend, Length: 224, dtype: bool


### Correlations and Visualizations

In [112]:
mock_df.head()

,observation_time,scheduled_departures,wind_speed,visibility,temperature,precipitation,weather_condition,avg_departure_delay_min,day_of_week,is_weekend
0,2024-01-09 06:00:00,37,19,4.1,26,0.03,Snow,47.9,Tuesday,False
1,2024-01-09 09:00:00,57,8,10.0,46,Trace,Clear,23.8,Tuesday,False
2,2024-01-09 12:00:00,41,20,7.9,34,0.00,Cloudy,27.4,Tuesday,False
3,2024-01-09 15:00:00,50,13,9.0,32,0.00,Cloudy,22.0,Tuesday,False
5,2024-01-09 21:00:00,15,15,10.0,15,0.00,Clear,15.0,Tuesday,False


In [117]:
correlation_matrix = mock_df[['wind_speed', 'temperature', 'avg_departure_delay_min']].corr()
correlation_matrix

,wind_speed,temperature,avg_departure_delay_min
wind_speed,1.000000,0.001349,0.170174
temperature,0.001349,1.000000,0.006797
avg_departure_delay_min,0.170174,0.006797,1.000000


# Week 4 Application: Dataset Exploration and Regression Setup

### The Data

In [118]:
flight_df = pd.read_csv('ord_hourly_flights_2022_2025.csv', low_memory=False)
weather_df = pd.read_csv('weather_data/ord_weather_2022_2025', low_memory=False)

In [119]:
display(flight_df.head())
print(flight_df.shape)
display(weather_df.head())
print(weather_df.shape)

,FlightDate,arrival_hour,scheduled_flights,airlines,origins,avg_distance,avg_scheduled_flight_time,avg_departure_delay,avg_arrival_delay,median_arrival_delay,delayed_15_rate,cancelled_rate,diverted_rate,weather_delay_minutes,carrier_delay_minutes,nas_delay_minutes,late_aircraft_delay_minutes
0,2022-01-01,0,8,4,8,1249.625000,191.750000,183.800,182.80,20.0,0.500000,0.375000,0.0,474.0,72.0,21.0,349.0
1,2022-01-01,1,1,1,1,1744.000000,245.000000,NaN,NaN,NaN,0.000000,1.000000,0.0,0.0,0.0,0.0,0.0
2,2022-01-01,4,2,2,2,1581.500000,220.500000,NaN,NaN,NaN,0.000000,1.000000,0.0,0.0,0.0,0.0,0.0
3,2022-01-01,5,13,4,8,1680.000000,231.153846,42.625,31.25,8.5,0.307692,0.384615,0.0,22.0,40.0,0.0,196.0
4,2022-01-01,6,7,3,5,2800.285714,346.571429,43.000,38.00,3.0,0.285714,0.285714,0.0,0.0,191.0,17.0,0.0


(28681, 17)


,Unnamed: 0,DATE,REPORT_TYPE,WND,CIG,VIS,TMP,DEW,SLP,AA1,AU1,AW1,GA1,MW1,OC1
0,0,2022-01-01 00:00:00,FM-12,"030,1,N,0041,1","99999,9,9,N","002000,1,9,9","+0056,1","+0039,1","10038,1",NaN,NaN,NaN,"99,9,+00075,1,99,9","10,1",NaN
1,1,2022-01-01 00:06:00,FM-16,"030,5,N,0046,5","00122,5,M,N","002012,5,N,5","+0056,5","+0039,5","99999,9",NaN,"0,0,00,1,0,1,5","10,5","08,5,+00122,5,99,9",NaN,NaN
2,2,2022-01-01 00:30:00,FM-16,"350,5,N,0057,5","00122,5,M,N","003219,5,N,5","+0056,5","+0039,5","99999,9",NaN,"0,0,00,1,0,1,5","10,5","08,5,+00122,5,99,9",NaN,NaN
3,3,2022-01-01 00:51:00,FM-15,"040,5,N,0051,5","00122,5,M,N","003219,5,N,5","+0050,5","+0033,5","10040,5","01,0000,9,5","0,0,00,1,0,1,5","10,5","08,5,+00122,5,99,9",NaN,"0082,5"
4,4,2022-01-01 01:51:00,FM-15,"020,5,N,0041,5","00122,5,M,N","003219,5,N,5","+0050,5","+0033,5","10044,5","01,0000,9,5","0,0,00,1,0,1,5","10,5","08,5,+00122,5,99,9",NaN,NaN


(49334, 15)


### Preliminary Cleanup

In [17]:
weather_df.columns = ['date', 'report_type', 'wind', 'ceiling_height', 'visibility', 
                      'dry_bulb_temp', 'dew_point_temp', 'sea_level_pressure', 
                      'liquid_precipitation', 'present_weather', 'auto_condition_code',
                      'sky_cover', 'manual_condition_code', 'wind_gust']

# changing abbreviations

In [19]:
flight_df.head()

,FlightDate,arrival_hour,scheduled_flights,airlines,origins,avg_distance,avg_scheduled_flight_time,avg_departure_delay,avg_arrival_delay,median_arrival_delay,delayed_15_rate,cancelled_rate,diverted_rate,weather_delay_minutes,carrier_delay_minutes,nas_delay_minutes,late_aircraft_delay_minutes
0,2022-01-01,0,8,4,8,1249.625000,191.750000,183.800,182.80,20.0,0.500000,0.375000,0.0,474.0,72.0,21.0,349.0
1,2022-01-01,1,1,1,1,1744.000000,245.000000,NaN,NaN,NaN,0.000000,1.000000,0.0,0.0,0.0,0.0,0.0
2,2022-01-01,4,2,2,2,1581.500000,220.500000,NaN,NaN,NaN,0.000000,1.000000,0.0,0.0,0.0,0.0,0.0
3,2022-01-01,5,13,4,8,1680.000000,231.153846,42.625,31.25,8.5,0.307692,0.384615,0.0,22.0,40.0,0.0,196.0
4,2022-01-01,6,7,3,5,2800.285714,346.571429,43.000,38.00,3.0,0.285714,0.285714,0.0,0.0,191.0,17.0,0.0


In [18]:
weather_df.head()

,date,report_type,wind,ceiling_height,visibility,dry_bulb_temp,dew_point_temp,sea_level_pressure,liquid_precipitation,present_weather,auto_condition_code,sky_cover,manual_condition_code,wind_gust
0,2022-01-01 00:00:00,FM-12,"030,1,N,0041,1","99999,9,9,N","002000,1,9,9","+0056,1","+0039,1","10038,1",NaN,NaN,NaN,"99,9,+00075,1,99,9","10,1",NaN
1,2022-01-01 00:06:00,FM-16,"030,5,N,0046,5","00122,5,M,N","002012,5,N,5","+0056,5","+0039,5","99999,9",NaN,"0,0,00,1,0,1,5","10,5","08,5,+00122,5,99,9",NaN,NaN
2,2022-01-01 00:30:00,FM-16,"350,5,N,0057,5","00122,5,M,N","003219,5,N,5","+0056,5","+0039,5","99999,9",NaN,"0,0,00,1,0,1,5","10,5","08,5,+00122,5,99,9",NaN,NaN
3,2022-01-01 00:51:00,FM-15,"040,5,N,0051,5","00122,5,M,N","003219,5,N,5","+0050,5","+0033,5","10040,5","01,0000,9,5","0,0,00,1,0,1,5","10,5","08,5,+00122,5,99,9",NaN,"0082,5"
4,2022-01-01 01:51:00,FM-15,"020,5,N,0041,5","00122,5,M,N","003219,5,N,5","+0050,5","+0033,5","10044,5","01,0000,9,5","0,0,00,1,0,1,5","10,5","08,5,+00122,5,99,9",NaN,NaN


### What do we need to do?

1. Convert weather dates to datetime objects
2. Aggregate weather by hour like flight data
3. Clean weather column values for valid regression
4. Choose regression features and analyze useful ones